# 01 — Inventário de imagens

Objetivo: identificar imagens, dimensões, hashes e duplicatas antes de recortes. Entradas: fixtures sintéticas por padrão; em modo real, `TCC_RAW` indica pasta ou ZIP. Saídas: inventário, exclusões e resumo. Pré-requisitos: ambiente do README. O inventário não confirma classes ou independência biológica.

Execução: na raiz, `python -m jupyter lab notebooks`. Execute de cima para baixo. Variáveis: `TCC_ROOT`, `TCC_RUN_MODE=synthetic|real`, `TCC_MANIFEST`, `TCC_ANNOTATIONS`, `TCC_SPLIT`. Colab pode definir `TCC_ROOT` para a cópia do repositório e instalar as dependências em runtime Python compatível.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path(os.environ.get("TCC_ROOT", Path.cwd())).resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Defina TCC_ROOT para a raiz do repositório."
sys.path.insert(0, str(ROOT))
MODE = os.environ.get("TCC_RUN_MODE", "synthetic")
assert MODE in {"synthetic", "real"}
from scripts.notebook_support import prepare_synthetic, SyntheticClassifier
from src import data_protocol as dp
if MODE == "synthetic":
    OUTPUT, rows = prepare_synthetic(ROOT)
    MANIFEST = OUTPUT / "images.csv"
    ANNOTATIONS = OUTPUT / "synthetic_annotations.json"
else:
    OUTPUT = ROOT / "experiments" / "runs" / "notebook_real"
    OUTPUT.mkdir(parents=True, exist_ok=True)
    MANIFEST = Path(os.environ.get("TCC_MANIFEST", ROOT / "data/splits/luiz/images.csv"))
    ANNOTATIONS = Path(os.environ.get("TCC_ANNOTATIONS", ROOT / "data/annotations/luiz_pending.json"))
print({"modo": MODE, "raiz": str(ROOT), "manifesto": str(MANIFEST)})


In [ ]:
if MODE == "synthetic":
    report = dp.build_inventory([OUTPUT / "raw"])
    print(report["summary"])
else:
    raw = os.environ.get("TCC_RAW")
    assert raw, "Defina TCC_RAW para pasta ou ZIP real."
    subprocess.run([sys.executable, str(ROOT / "scripts/prepare_data.py"), "inventory",
                    "--input", raw, "--output-dir", str(OUTPUT / "inventory")], check=True, cwd=ROOT)


As fixtures são formas geométricas, sem material biológico. No lote real Luiz já auditado, os grupos são provisoriamente imagens originais; a confirmação de animal/lâmina continua pendente.